# # Cell 1 - Load data + model từ Unity Catalog


In [0]:
%pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 MB 156.2 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:

import pandas as pd
import numpy as np
import mlflow.xgboost

# Load feature data
fe = pd.read_parquet("/Volumes/workspace/default/retail_data/customer_features_engineered.parquet")

feature_cols = [
    "total_spent", "avg_unit_price", "num_invoices", "num_line_items",
    "num_unique_products", "num_countries", "recency_days", "active_days",
    "purchase_frequency", "avg_basket_value",
    "log_total_spent", "log_num_invoices", "log_num_unique_products", "log_avg_basket_value"
]

X = fe[feature_cols]

# Load model từ Unity Catalog
model_uri = "models:/workspace.default.retail-customer-segmentation/1"
model = mlflow.xgboost.load_model(model_uri)

print("Model loaded!")
print(f"Scoring {len(X):,} customers...")

Model loaded!
Scoring 5,878 customers...


# Cell 2 - Batch Scoring

In [0]:

# Predict
y_pred = model.predict(X)

# Map integer → segment name thủ công
label_map = {0: "At Risk", 1: "Champions", 2: "Loyal", 3: "Potential"}
segments = [label_map[p] for p in y_pred]

# Gắn kết quả vào dataframe
output = fe[["Customer ID"]].copy()
output["predicted_segment"] = segments
output["scoring_date"] = pd.Timestamp.now().date()

print("=== SCORING RESULTS ===")
print(output["predicted_segment"].value_counts())
print(f"\nSample output:")
print(output.head(10))

# Save
output.to_parquet("/Volumes/workspace/default/retail_data/customer_segments_scored.parquet", index=False)
output.to_csv("/Volumes/workspace/default/retail_data/customer_segments_scored.csv", index=False)
print("\nSaved!")

=== SCORING RESULTS ===
predicted_segment
Potential    1684
Champions    1681
Loyal        1422
At Risk      1091
Name: count, dtype: int64

Sample output:
  Customer ID predicted_segment scoring_date
0       17764             Loyal   2026-04-10
1       15031         Champions   2026-04-10
2       15785         Champions   2026-04-10
3       18041         Champions   2026-04-10
4       14344         Champions   2026-04-10
5       13394         Champions   2026-04-10
6       17476         Potential   2026-04-10
7       13768         Potential   2026-04-10
8       13921         Potential   2026-04-10
9       14970         Champions   2026-04-10

Saved!


##  Batch Scoring Pipeline

### Output:
| Segment | Count |
|---|---|
| Potential | 1,684 |
| Champions | 1,681 |
| Loyal | 1,422 |
| At Risk | 1,091 |
| **Total** | **5,878** |

### Pipeline flow:
1. Load feature data từ Unity Catalog Volume
2. Load model từ MLflow Registry (`version 1`)
3. Predict segment cho toàn bộ 5,878 customers
4. Save output ra cả `.parquet` và `.csv`

### Output schema:
- `Customer ID` - định danh customer
- `predicted_segment` - segment được predict
- `scoring_date` - ngày chạy scoring → quan trọng cho tracking theo thời gian

### Kết quả so sánh với rule-based labels:
Phân bổ gần như giống hệt Phase 5 (sai lệch tối đa 2 customers/segment)
→ model học lại được rules rất chính xác, F1=0.9984 được confirm trên full data.